# Figuring out the dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Define Local Path

In [ ]:
import os

# TODO: Fill in the Google Drive path where you uploaded the CW_folder_UG
# Example: GOOGLE_DRIVE_PATH_AFTER_MYDRIVE = 'Colab Notebooks/Computer Vision/CW_folder_UG'

GOOGLE_DRIVE_PATH_AFTER_MYDRIVE = 'Colab Notebooks/CW_Folder_UG'
GOOGLE_DRIVE_PATH = os.path.join('drive', 'My Drive', GOOGLE_DRIVE_PATH_AFTER_MYDRIVE)
print(os.listdir(GOOGLE_DRIVE_PATH))

['CW_Dataset', 'Models', 'Personal_Dataset', 'Code', 'work.ipynb', 'test_function.ipynb', 'Copy of work.ipynb']


## UnZip Dataset and load to drive


In [ ]:
# Identify path to zipped dataset
zip_path = os.path.join(GOOGLE_DRIVE_PATH, 'CW_Dataset/CW_Dataset.zip')

# Copy it to Colab
!cp '{zip_path}' .

# Unzip it
!yes|unzip -q CW_Dataset.zip

# Delete zipped version from Colab (not from Drive)
!rm CW_Dataset.zip

In [ ]:
import os

print(os.listdir())
print(os.listdir("CW_Dataset"))
print(os.listdir("CW_Dataset/train")[:5])
print(os.listdir("CW_Dataset/test")[:5])

['.config', 'CW_Dataset', 'drive', 'sample_data']
['test', 'README.txt', 'train']
['train_10292.jpg', 'train_5632.jpg', 'train_6864.jpg', 'train_1296.jpg', 'train_12304.jpg']
['test_0409.jpg', 'test_0184.jpg', 'test_0007.jpg', 'test_0684.jpg', 'test_0535.jpg']


In [ ]:
import os
import sys
CODE_PATH = os.path.join(GOOGLE_DRIVE_PATH, 'Code')
sys.path.append(CODE_PATH)

from feature_set_extractor import HOGExtractor, BoVWExtractor, CNNExtractor
from dataset_analysis import analyse_image_sizes, print_stats
from preprocessor import AgeDatasetPreprocessor
from ageSVM import AgeSVM
from ageMLP import AgeMLP
from ageCNN import AgeCNN

DATASET_PATH = '/content/CW_Dataset'

## Discover Image sizes


In [ ]:
train_sizes, train_failed = analyse_image_sizes("train")
test_sizes, test_failed = analyse_image_sizes("test")


print_stats("train", train_sizes, train_failed)
print_stats("test", test_sizes, test_failed)


==== train Set ====
Total images: 13300
Failed 0

 Min & MaX
Width: 100 -> 200
Height: 100 -> 200

==== test Set ====
Total images: 850
Failed 0

 Min & MaX
Width: 100 -> 200
Height: 100 -> 200


# TESTING

In [ ]:
hog_extractor = HOGExtractor()
agd = AgeDatasetPreprocessor("/content/CW_Dataset", extractor=hog_extractor)
X_train, X_val, y_train, y_val = agd.load_train(limit_ratio=0.1)
X_test, y_test = agd.load_test(limit_ratio=0.1)

In [ ]:
configs = [
    # Linear
    AgeSVM(kernel='linear', C=1),
    AgeSVM(kernel='linear', C=10),

    # rbf
    AgeSVM(kernel='rbf', C=1,  gamma='scale'),
    AgeSVM(kernel='rbf', C=1,  gamma='auto'),
    AgeSVM(kernel='rbf', C=10, gamma='scale'),
    AgeSVM(kernel='rbf', C=10, gamma='auto'),
]

for model in configs:
  model.fit(X_train, y_train)
  print(type(model.model))
  model.evaluate(X_val, y_val)

<class 'sklearn.svm._classes.SVC'>
Accuracy: 0.6616541353383458

Classification Report:

              precision    recall  f1-score   support

           0       0.80      0.80      0.80        25
           1       0.71      0.70      0.70        56
           2       0.52      0.52      0.52        29
           3       0.58      0.61      0.60        23

    accuracy                           0.66       133
   macro avg       0.65      0.66      0.65       133
weighted avg       0.66      0.66      0.66       133

<class 'sklearn.svm._classes.SVC'>
Accuracy: 0.6616541353383458

Classification Report:

              precision    recall  f1-score   support

           0       0.80      0.80      0.80        25
           1       0.71      0.70      0.70        56
           2       0.52      0.52      0.52        29
           3       0.58      0.61      0.60        23

    accuracy                           0.66       133
   macro avg       0.65      0.66      0.65       133
weighte

In [ ]:
hog_extractor = HOGExtractor()
agd = AgeDatasetPreprocessor("/content/CW_Dataset", extractor=hog_extractor)
X_train, X_val, y_train, y_val = agd.load_train(limit_ratio=1)
X_test, y_test = agd.load_test(limit_ratio=1)

In [ ]:
svm = AgeSVM(kernel='linear', C=1)
svm.fit(X_train, y_train)
svm.evaluate(X_val, y_val)
svm.evaluate(X_test, y_test)

Accuracy: 0.630827067669173

Classification Report:

              precision    recall  f1-score   support

           0       0.75      0.83      0.79       240
           1       0.69      0.69      0.69       550
           2       0.44      0.44      0.44       300
           3       0.60      0.54      0.57       240

    accuracy                           0.63      1330
   macro avg       0.62      0.62      0.62      1330
weighted avg       0.63      0.63      0.63      1330

Accuracy: 0.66

Classification Report:

              precision    recall  f1-score   support

           0       0.76      0.89      0.82       150
           1       0.73      0.73      0.73       350
           2       0.48      0.47      0.47       200
           3       0.60      0.53      0.57       150

    accuracy                           0.66       850
   macro avg       0.64      0.65      0.65       850
weighted avg       0.65      0.66      0.66       850



0.66

In [ ]:
from joblib import dump
dump(svm, os.path.join(GOOGLE_DRIVE_PATH, 'Models', 'svm_best.joblib'))
print("Model saved.")

Model saved.


In [ ]:
bovw_extractor = BoVWExtractor(num_clusters=100)
pre = AgeDatasetPreprocessor("/content/CW_Dataset", extractor=bovw_extractor)

X_train, X_val, y_train, y_val = pre.load_train(limit_ratio=0.3)
X_test, y_test = pre.load_test(limit_ratio=0.3)

In [ ]:
configs = [
    # shallow
    AgeMLP(hidden_layers=(128,),        max_iter=250),
    AgeMLP(hidden_layers=(256,),        max_iter=250),

    # two layers
    AgeMLP(hidden_layers=(256, 128),    max_iter=250),
    AgeMLP(hidden_layers=(512, 256),    max_iter=250),

    # three layers
    AgeMLP(hidden_layers=(512, 256, 128), max_iter=250),
]

for model in configs:
    model.fit(X_train, y_train)
    print(f"Config: {model.model.hidden_layer_sizes}")
    model.evaluate(X_val, y_val)

Config: (128,)
Accuracy: 0.46115288220551376

Classification Report:

              precision    recall  f1-score   support

           0       0.52      0.49      0.50        72
           1       0.54      0.60      0.57       169
           2       0.29      0.29      0.29        87
           3       0.40      0.32      0.36        71

    accuracy                           0.46       399
   macro avg       0.44      0.42      0.43       399
weighted avg       0.46      0.46      0.46       399

Config: (256,)
Accuracy: 0.48370927318295737

Classification Report:

              precision    recall  f1-score   support

           0       0.50      0.44      0.47        72
           1       0.54      0.65      0.59       169
           2       0.34      0.29      0.31        87
           3       0.45      0.37      0.40        71

    accuracy                           0.48       399
   macro avg       0.46      0.44      0.44       399
weighted avg       0.47      0.48      0.47  

In [ ]:
bovw_extractor = BoVWExtractor(num_clusters=100)
pre = AgeDatasetPreprocessor("/content/CW_Dataset", extractor=bovw_extractor)

X_train, X_val, y_train, y_val = pre.load_train(limit_ratio=1)
X_test, y_test = pre.load_test(limit_ratio=1)

In [ ]:
MLP = AgeMLP(hidden_layers=(256,),max_iter=250)
MLP.fit(X_train, y_train)
MLP.evaluate(X_val, y_val)
MLP.evaluate(X_test, y_test)

Accuracy: 0.4218045112781955

Classification Report:

              precision    recall  f1-score   support

           0       0.44      0.41      0.43       240
           1       0.52      0.54      0.53       550
           2       0.26      0.25      0.26       300
           3       0.36      0.37      0.36       240

    accuracy                           0.42      1330
   macro avg       0.40      0.39      0.39      1330
weighted avg       0.42      0.42      0.42      1330

Accuracy: 0.41058823529411764

Classification Report:

              precision    recall  f1-score   support

           0       0.45      0.48      0.47       150
           1       0.53      0.52      0.52       350
           2       0.27      0.28      0.27       200
           3       0.28      0.27      0.27       150

    accuracy                           0.41       850
   macro avg       0.38      0.39      0.38       850
weighted avg       0.41      0.41      0.41       850



0.41058823529411764

In [ ]:
from joblib import dump
dump(MLP, os.path.join(GOOGLE_DRIVE_PATH, 'Models', 'mlp_best.joblib'))
print("Model saved.")

Model saved.


In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True
Tesla T4


In [ ]:
cnn_extractor = CNNExtractor()
pre = AgeDatasetPreprocessor("/content/CW_Dataset", extractor=cnn_extractor)
X_train, X_val, y_train, y_val = pre.load_train(limit_ratio=0.3)
X_test, y_test = pre.load_test(limit_ratio=0.3)

In [ ]:
configs = [
    AgeCNN(epochs=10, filters=(32, 64, 128),  dropout=0.5),  # baseline
    AgeCNN(epochs=10, filters=(64, 128, 256), dropout=0.5),  # wider
    AgeCNN(epochs=10, filters=(32, 64, 128),  dropout=0.3),  # less dropout
    AgeCNN(epochs=15, filters=(64, 128, 256), dropout=0.3),  # wider + more epochs
]

for i, model in enumerate(configs):
    print(f"\n--- Config {i+1} ---")
    model.fit(X_train, y_train)
    model.evaluate(X_val, y_val)


--- Config 1 ---
Epoch 1/10, Loss: 337.8490
Epoch 2/10, Loss: 116.8800
Epoch 3/10, Loss: 93.8899
Epoch 4/10, Loss: 87.2091
Epoch 5/10, Loss: 78.0215
Epoch 6/10, Loss: 75.6339
Epoch 7/10, Loss: 69.9695
Epoch 8/10, Loss: 64.9061
Epoch 9/10, Loss: 55.2758
Epoch 10/10, Loss: 54.9114
Accuracy: 0.7017543859649122

Classification Report:

              precision    recall  f1-score   support

           0       0.83      0.76      0.80        72
           1       0.72      0.88      0.79       169
           2       0.55      0.24      0.34        87
           3       0.62      0.77      0.69        71

    accuracy                           0.70       399
   macro avg       0.68      0.67      0.65       399
weighted avg       0.69      0.70      0.68       399


--- Config 2 ---
Epoch 1/10, Loss: 1134.3023
Epoch 2/10, Loss: 140.6705
Epoch 3/10, Loss: 148.6816
Epoch 4/10, Loss: 147.7757
Epoch 5/10, Loss: 146.9440
Epoch 6/10, Loss: 144.0651
Epoch 7/10, Loss: 138.6635
Epoch 8/10, Loss: 138.

In [ ]:
cnn_extractor = CNNExtractor()
pre = AgeDatasetPreprocessor("/content/CW_Dataset", extractor=cnn_extractor)
X_train, X_val, y_train, y_val = pre.load_train(limit_ratio=1.0)
X_test, y_test = pre.load_test(limit_ratio=1.0)

In [ ]:
best_cnn = AgeCNN(epochs=10, filters=(32, 64, 128), dropout=0.5)
best_cnn.fit(X_train, y_train)
best_cnn.evaluate(X_val, y_val)
best_cnn.evaluate(X_test, y_test)

Epoch 1/10, Loss: 894.3211
Epoch 2/10, Loss: 430.0247
Epoch 3/10, Loss: 366.8111
Epoch 4/10, Loss: 290.8452
Epoch 5/10, Loss: 266.2508
Epoch 6/10, Loss: 248.0783
Epoch 7/10, Loss: 233.1663
Epoch 8/10, Loss: 219.6716
Epoch 9/10, Loss: 207.8436
Epoch 10/10, Loss: 193.0428
Accuracy: 0.7

Classification Report:

              precision    recall  f1-score   support

           0       0.88      0.82      0.84       240
           1       0.75      0.79      0.77       550
           2       0.50      0.56      0.53       300
           3       0.69      0.56      0.62       240

    accuracy                           0.70      1330
   macro avg       0.70      0.68      0.69      1330
weighted avg       0.71      0.70      0.70      1330

Accuracy: 0.7294117647058823

Classification Report:

              precision    recall  f1-score   support

           0       0.91      0.82      0.86       150
           1       0.77      0.82      0.79       350
           2       0.55      0.61     

0.7294117647058823

In [ ]:
from joblib import dump
dump(best_cnn, os.path.join(GOOGLE_DRIVE_PATH, 'Models', 'cnn_best.joblib'))
print("Model saved.")

Model saved.
